# Multi-Modal ICU Discharge Time Prediction

**Task**: Predict `time_to_icu_discharge_hours` using a fusion of:
- **Demographics** (age, marital_status, language_group)
- **Radiology notes** encoded by `Qwen/Qwen3.5-0.8B` (frozen, mean-pooled)
- **Time series** from `data_ts_3h.csv` encoded by a Transformer Encoder

**Loss**: `nn.MSELoss` on `time_to_icu_discharge_hours`

In [ ]:
# Uncomment to install: !pip install transformers accelerate
import math, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Configuration ──────────────────────────────────────────────────────────
DATA_DIR, CHECKPOINT_PATH = 'data/', 'best_model.pt'
DISCHARGE_FILE = DATA_DIR + 'data_feature_discharge.csv'
DEATH_FILE     = DATA_DIR + 'data_feature_death.csv'
TS_FILE        = DATA_DIR + 'data_ts_3h.csv'

TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED = 0.70, 0.15, 0.15, 42

# Time series: first 57 bins (hour_3h -3 to 165 covers 7 days)
MAX_TS_STEPS = 57
TS_FEATURES = [
    'hr', 'map', 'rr', 'spo2', 'temp', 'gcs',
    'lactate', 'creatinine', 'bilirubin', 'platelets', 'wbc',
    'sodium', 'bun', 'inr',
    'urine_output', 'vasopressor_dose', 'fluid_input', 'ventilation_flag',
    'sofa_resp', 'sofa_cardio', 'sofa_renal', 'sofa_liver', 'sofa_coag', 'sofa_cns',
    'max24_sofa_resp', 'max24_sofa_cardio', 'max24_sofa_renal',
    'max24_sofa_liver', 'max24_sofa_coag', 'max24_sofa_cns'
]
TS_INPUT_DIM = len(TS_FEATURES)  # 30

TS_EMBED_DIM, TS_NHEAD, TS_NUM_LAYERS = 64, 4, 2
TS_DIM_FF, TS_DROPOUT = 128, 0.1

QWEN_MODEL_NAME = 'Qwen/Qwen3.5-0.8B'
TEXT_PROJ_DIM, TEXT_MAX_LENGTH = 128, 256

DEMO_EMBED_DIM, DEMO_HIDDEN_DIM = 16, 64
FUSION_HIDDEN_DIM = 256
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
BATCH_SIZE, NUM_EPOCHS, SCHEDULER_PATIENCE = 32, 30, 5

print(f'TS input dim : {TS_INPUT_DIM}')
print(f'Fusion dim   : {DEMO_HIDDEN_DIM}+{TS_EMBED_DIM}+{TEXT_PROJ_DIM} = {DEMO_HIDDEN_DIM+TS_EMBED_DIM+TEXT_PROJ_DIM}')

In [ ]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────
df_discharge = pd.read_csv(DISCHARGE_FILE)
df_death     = pd.read_csv(DEATH_FILE)
df_ts        = pd.read_csv(TS_FILE)

print(f'Discharge : {df_discharge.shape}')
print(f'Death     : {df_death.shape}')
print(f'TS        : {df_ts.shape}')
print(f'Unique patients (discharge): {df_discharge["stay_id"].nunique()}')
print(f'Unique patients (TS)       : {df_ts["stay_id"].nunique()}')
print('\nTarget stats:')
print(df_discharge['time_to_icu_discharge_hours'].describe())

# Confirm overlap between data sources
discharge_ids = set(df_discharge['stay_id'])
death_ids     = set(df_death['stay_id'])
ts_ids        = set(df_ts['stay_id'])
print(f'\nDischarge intersect Death : {len(discharge_ids & death_ids)}')
print(f'Discharge intersect TS    : {len(discharge_ids & ts_ids)}')

# Restrict to patients present in both discharge and TS tables
common_ids   = discharge_ids & ts_ids
df_discharge = df_discharge[df_discharge['stay_id'].isin(common_ids)].reset_index(drop=True)
assert df_discharge['stay_id'].is_unique, 'stay_id must be unique'
print(f'Final cohort : {len(df_discharge)} patients')

In [ ]:
# ── Cell 3: Train / Val / Test Split (70 / 15 / 15 by stay_id) ──────────
# Split on stay_id to prevent patient-level data leakage.
all_ids = df_discharge['stay_id'].values
ids_trainval, ids_test = train_test_split(
    all_ids, test_size=TEST_RATIO, random_state=RANDOM_SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)  # ~0.176
ids_train, ids_val = train_test_split(
    ids_trainval, test_size=val_frac, random_state=RANDOM_SEED)

print(f'Train : {len(ids_train):5d} ({len(ids_train)/len(all_ids)*100:.1f}%)')
print(f'Val   : {len(ids_val):5d} ({len(ids_val)/len(all_ids)*100:.1f}%)')
print(f'Test  : {len(ids_test):5d} ({len(ids_test)/len(all_ids)*100:.1f}%)')

split_map = {}
for sid in ids_train: split_map[sid] = 'train'
for sid in ids_val:   split_map[sid] = 'val'
for sid in ids_test:  split_map[sid] = 'test'
df_discharge['split'] = df_discharge['stay_id'].map(split_map)
train_mask = df_discharge['split'] == 'train'

In [ ]:
# ── Cell 4: Demographic Feature Preprocessing ────────────────────────────
# age + marital_status appear in both files; language_group is discharge-only.
# Using discharge file as the single source of truth for all demographics.
DEMO_COLS_CAT = ['marital_status', 'language_group']

for col in DEMO_COLS_CAT:
    df_discharge[col] = df_discharge[col].fillna('UNKNOWN').astype(str)

# Fit encoders on TRAINING split only (prevents leakage)
label_encoders = {}
for col in DEMO_COLS_CAT:
    le = LabelEncoder()
    le.fit(df_discharge.loc[train_mask, col])
    known = set(le.classes_)
    df_discharge[col + '_enc'] = df_discharge[col].apply(
        lambda x: int(le.transform([x])[0]) if x in known else -1)
    label_encoders[col] = le
    print(f'{col}: {len(le.classes_)} classes -> {list(le.classes_)}')

# +1 reserves index 0 for unknown / padding in nn.Embedding
marital_vocab = len(label_encoders['marital_status'].classes_) + 1
lang_vocab    = len(label_encoders['language_group'].classes_)  + 1
print(f'marital_vocab={marital_vocab}, lang_vocab={lang_vocab}')

# Normalise age with TRAINING statistics only
age_mean = df_discharge.loc[train_mask, 'age'].mean()
age_std  = df_discharge.loc[train_mask, 'age'].std()
df_discharge['age_norm'] = (
    df_discharge['age'].fillna(age_mean) - age_mean) / (age_std + 1e-8)
print(f'Age: mean={age_mean:.1f} y, std={age_std:.1f} y')

In [ ]:
# ── Cell 5: Time Series Preprocessing ───────────────────────────────────
# Keep first MAX_TS_STEPS bins (hour_3h from -3 to 165 = 7 days).
max_hour   = (MAX_TS_STEPS - 1) * 3 - 3  # = 165
df_ts_filt = df_ts[df_ts['hour_3h'] <= max_hour].copy()
print(f'TS rows after time filter: {len(df_ts_filt)}')

# Normalise with TRAINING patient statistics only
train_ts = df_ts_filt[df_ts_filt['stay_id'].isin(ids_train)]
ts_means = train_ts[TS_FEATURES].mean()
ts_stds  = train_ts[TS_FEATURES].std().replace(0, 1)
df_ts_filt[TS_FEATURES] = (
    (df_ts_filt[TS_FEATURES] - ts_means) / ts_stds).fillna(0.0)

# Build lookup: stay_id -> ndarray (T_actual, 30)
ts_lookup = {}
for sid, grp in df_ts_filt.groupby('stay_id'):
    arr = grp.sort_values('hour_3h')[TS_FEATURES].values.astype(np.float32)
    ts_lookup[sid] = arr[:MAX_TS_STEPS]  # truncate to first N bins

seq_lens = [v.shape[0] for v in ts_lookup.values()]
print(f'Patients with TS: {len(ts_lookup)}')
print(f'Seq len min/median/max: {min(seq_lens)} / {int(np.median(seq_lens))} / {max(seq_lens)}')

In [ ]:
# ── Cell 6: Radiology Note Tokenization ──────────────────────────────────
print(f'Loading tokenizer: {QWEN_MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print('Set pad_token = eos_token')

def tokenize_note(text):
    """Returns tokenised dict or None for missing/empty notes."""
    if pd.isna(text) or str(text).strip() == '':
        return None
    return tokenizer(
        str(text), max_length=TEXT_MAX_LENGTH,
        truncation=True, padding='max_length', return_tensors='pt')

# Pre-tokenise once to speed up DataLoader batches
note_tokens = {}
for _, row in tqdm(df_discharge.iterrows(), total=len(df_discharge), desc='Tokenising'):
    note_tokens[row['stay_id']] = tokenize_note(row['radiology_note_text'])

missing = sum(1 for v in note_tokens.values() if v is None)
print(f'Missing notes: {missing} / {len(note_tokens)}')

In [ ]:
# ── Cell 7: ICUDataset ────────────────────────────────────────────────────
class ICUDataset(Dataset):
    """
    Multi-modal Dataset. Each sample returns:
        age, marital_enc, lang_enc              demographics
        ts_seq (MAX_TS_STEPS,30), ts_mask       time series
        input_ids, attention_mask (L,), has_text  radiology text
        target (1,)  time_to_icu_discharge_hours
    """
    def __init__(self, stay_ids, df, ts_lookup, note_tokens):
        self.stay_ids    = stay_ids
        self.df          = df.set_index('stay_id')
        self.ts_lookup   = ts_lookup
        self.note_tokens = note_tokens

    def __len__(self): return len(self.stay_ids)

    def __getitem__(self, idx):
        sid = self.stay_ids[idx]
        row = self.df.loc[sid]

        # Demographics (clamp -1 to 0 = unknown embedding)
        age_t     = torch.tensor([row['age_norm']], dtype=torch.float32)
        marital_t = torch.tensor(max(int(row['marital_status_enc']), 0), dtype=torch.long)
        lang_t    = torch.tensor(max(int(row['language_group_enc']),  0), dtype=torch.long)

        # Time series (pad to MAX_TS_STEPS; ts_mask True = padding)
        if sid in self.ts_lookup:
            arr = self.ts_lookup[sid]
            T   = arr.shape[0]
            if T >= MAX_TS_STEPS:
                ts_t    = torch.from_numpy(arr[:MAX_TS_STEPS])
                ts_mask = torch.zeros(MAX_TS_STEPS, dtype=torch.bool)
            else:
                pad     = np.zeros((MAX_TS_STEPS - T, TS_INPUT_DIM), dtype=np.float32)
                ts_t    = torch.from_numpy(np.concatenate([arr, pad], axis=0))
                ts_mask = torch.zeros(MAX_TS_STEPS, dtype=torch.bool)
                ts_mask[T:] = True
        else:
            ts_t    = torch.zeros(MAX_TS_STEPS, TS_INPUT_DIM, dtype=torch.float32)
            ts_mask = torch.ones(MAX_TS_STEPS, dtype=torch.bool)

        # Radiology note tokens
        tokens = self.note_tokens.get(sid)
        if tokens is not None:
            input_ids_t = tokens['input_ids'].squeeze(0)
            attn_mask_t = tokens['attention_mask'].squeeze(0)
            has_text_t  = torch.tensor([1.0], dtype=torch.float32)
        else:
            input_ids_t = torch.zeros(TEXT_MAX_LENGTH, dtype=torch.long)
            attn_mask_t = torch.zeros(TEXT_MAX_LENGTH, dtype=torch.long)
            has_text_t  = torch.tensor([0.0], dtype=torch.float32)

        target_t = torch.tensor([row['time_to_icu_discharge_hours']], dtype=torch.float32)
        return {
            'age': age_t, 'marital_enc': marital_t, 'lang_enc': lang_t,
            'ts_seq': ts_t, 'ts_mask': ts_mask,
            'input_ids': input_ids_t, 'attention_mask': attn_mask_t,
            'has_text': has_text_t, 'target': target_t
        }

print('ICUDataset defined.')

In [ ]:
# ── Cell 8: DataLoaders ──────────────────────────────────────────────────
train_dataset = ICUDataset(ids_train, df_discharge, ts_lookup, note_tokens)
val_dataset   = ICUDataset(ids_val,   df_discharge, ts_lookup, note_tokens)
test_dataset  = ICUDataset(ids_test,  df_discharge, ts_lookup, note_tokens)

# num_workers=0 required on Windows
kw = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
train_loader = DataLoader(train_dataset, shuffle=True,  **kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **kw)
print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test : {len(test_dataset)} samples, {len(test_loader)} batches')

In [ ]:
# ── Cell 9: DemographicEncoder ───────────────────────────────────────────
class DemographicEncoder(nn.Module):
    """
    Encodes age (continuous), marital_status and language_group (categorical).
    Output: (B, DEMO_HIDDEN_DIM)
    """
    def __init__(self, marital_vocab_size, lang_vocab_size,
                 embed_dim=DEMO_EMBED_DIM, hidden_dim=DEMO_HIDDEN_DIM):
        super().__init__()
        # padding_idx=0 gives a zero embedding for unknown categories
        self.marital_embed = nn.Embedding(marital_vocab_size, embed_dim, padding_idx=0)
        self.lang_embed    = nn.Embedding(lang_vocab_size,    embed_dim, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(1 + embed_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU())

    def forward(self, age, marital_enc, lang_enc):
        m = self.marital_embed(marital_enc)   # (B, embed_dim)
        l = self.lang_embed(lang_enc)          # (B, embed_dim)
        return self.fc(torch.cat([age, m, l], dim=1))  # (B, hidden_dim)

print('DemographicEncoder defined.')

In [ ]:
# ── Cell 10: PositionalEncoding + TransformerTSEncoder ───────────────────
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (batch_first)."""
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerTSEncoder(nn.Module):
    """
    Encodes padded time series into a fixed vector.
    Pipeline: Linear(30->embed_dim) -> PosEnc -> TransformerEncoder -> mean-pool
    src_key_padding_mask: True = padding (ignored by attention).
    Output: (B, TS_EMBED_DIM)
    """
    def __init__(self, input_dim=TS_INPUT_DIM, embed_dim=TS_EMBED_DIM,
                 nhead=TS_NHEAD, num_layers=TS_NUM_LAYERS,
                 dim_feedforward=TS_DIM_FF, dropout=TS_DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.pos_enc    = PositionalEncoding(embed_dim, max_len=MAX_TS_STEPS + 5)
        enc = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)  # requires PyTorch >= 1.9
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)

    def forward(self, x, src_key_padding_mask=None):
        x = self.pos_enc(self.input_proj(x))
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        if src_key_padding_mask is not None:
            valid = (~src_key_padding_mask).unsqueeze(-1).float()
            return (x * valid).sum(1) / valid.sum(1).clamp(min=1)
        return x.mean(1)

print('TransformerTSEncoder defined.')

In [ ]:
# ── Cell 11: TextEncoder (Qwen, frozen) ──────────────────────────────────
class TextEncoder(nn.Module):
    """
    Frozen Qwen LM + trainable linear projection.
    Qwen is a causal/decoder-only LM: use mean-pool of last hidden states
    (NOT CLS token -- decoder-only LMs have no CLS token).
    Samples without a note are zeroed out via has_text gate.
    Output: (B, TEXT_PROJ_DIM)
    """
    def __init__(self, model_name=QWEN_MODEL_NAME, proj_dim=TEXT_PROJ_DIM):
        super().__init__()
        print(f'  Loading {model_name} ...')
        self.qwen = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        for p in self.qwen.parameters():
            p.requires_grad = False  # freeze all Qwen weights
        h = self.qwen.config.hidden_size
        self.proj    = nn.Linear(h, proj_dim)  # trainable projection
        self.dropout = nn.Dropout(0.1)
        print(f'  Qwen hidden_size={h} -> proj_dim={proj_dim}')

    def forward(self, input_ids, attention_mask, has_text):
        # torch.no_grad() prevents building autograd graph through frozen Qwen
        with torch.no_grad():
            out = self.qwen(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state  # (B, L, hidden_size)
        mask = attention_mask.unsqueeze(-1).float()
        emb  = (last * mask).sum(1) / mask.sum(1).clamp(min=1)  # mean-pool
        emb  = self.proj(self.dropout(emb))   # (B, proj_dim)
        return emb * has_text  # zero out missing notes

print('TextEncoder defined.')

In [ ]:
# ── Cell 12: FusionModel ─────────────────────────────────────────────────
class FusionModel(nn.Module):
    """
    Fuses all three modality encoders and predicts discharge time.
    Fusion dim: DEMO(64) + TS(64) + Text(128) = 256
    Head: 256->256->ReLU->Dropout(0.2)->64->ReLU->1
    Output: (B,) predicted time_to_icu_discharge_hours
    """
    def __init__(self, marital_vocab_size, lang_vocab_size):
        super().__init__()
        self.demo_enc = DemographicEncoder(marital_vocab_size, lang_vocab_size)
        self.ts_enc   = TransformerTSEncoder()
        self.text_enc = TextEncoder()
        fused = DEMO_HIDDEN_DIM + TS_EMBED_DIM + TEXT_PROJ_DIM  # 256
        self.head = nn.Sequential(
            nn.Linear(fused, FUSION_HIDDEN_DIM), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(FUSION_HIDDEN_DIM, 64), nn.ReLU(),
            nn.Linear(64, 1))

    def forward(self, b):
        d = self.demo_enc(b['age'], b['marital_enc'], b['lang_enc'])  # (B,64)
        t = self.ts_enc(b['ts_seq'], b['ts_mask'])                    # (B,64)
        x = self.text_enc(b['input_ids'], b['attention_mask'], b['has_text'])  # (B,128)
        return self.head(torch.cat([d, t, x], dim=1)).squeeze(1)      # (B,)

print('FusionModel defined.')

In [ ]:
# ── Cell 13: Instantiate Model & Sanity Check ────────────────────────────
model = FusionModel(marital_vocab_size=marital_vocab,
                    lang_vocab_size=lang_vocab).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')
print(f'Frozen (Qwen)    : {total - trainable:,}')

# Forward-pass sanity check
model.eval()
sb  = next(iter(train_loader))
sbg = {k: v.to(device) for k, v in sb.items()}
with torch.no_grad():
    out = model(sbg)
bs = sb['target'].shape[0]
assert out.shape == (bs,), f'Expected ({bs},), got {tuple(out.shape)}'
print(f'Output shape : {tuple(out.shape)}')
print(f'Output range : [{out.min().item():.1f}, {out.max().item():.1f}] h')
print('Sanity check passed!')

In [ ]:
# ── Cell 14: Training Setup ───────────────────────────────────────────────
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=SCHEDULER_PATIENCE, factor=0.5, verbose=True)
print('Criterion : nn.MSELoss')
print(f'Optimizer : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler : ReduceLROnPlateau (patience={SCHEDULER_PATIENCE}, factor=0.5)')

In [ ]:
# ── Cell 15: Training Loop ────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, device, train=True):
    """One epoch. Returns mean MSE per sample."""
    model.train(train)
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            bg      = {k: v.to(device) for k, v in batch.items()}
            preds   = model(bg)
            targets = bg['target'].squeeze(1)
            loss    = criterion(preds, targets)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * len(targets)
            n          += len(targets)
    return total_loss / n


train_losses, val_losses = [], []
best_val_loss = float('inf')
print(f'Training for {NUM_EPOCHS} epochs ...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    t = run_epoch(model, train_loader, optimizer, criterion, device, train=True)
    v = run_epoch(model, val_loader,   optimizer, criterion, device, train=False)
    scheduler.step(v)
    train_losses.append(t)
    val_losses.append(v)
    marker = ''
    if v < best_val_loss:
        best_val_loss = v
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        marker = '  <- best'
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train RMSE: {math.sqrt(t):7.2f} h | '
          f'Val RMSE: {math.sqrt(v):7.2f} h{marker}')

print(f'\nBest Val RMSE: {math.sqrt(best_val_loss):.2f} h  (saved: {CHECKPOINT_PATH})')

In [ ]:
# ── Cell 16: Learning Curve ───────────────────────────────────────────────
ep = range(1, len(train_losses) + 1)
plt.figure(figsize=(10, 4))
plt.plot(ep, [math.sqrt(l) for l in train_losses], label='Train RMSE')
plt.plot(ep, [math.sqrt(l) for l in val_losses],   label='Val RMSE')
plt.xlabel('Epoch'); plt.ylabel('RMSE (hours)')
plt.title('Training and Validation RMSE over Epochs')
plt.legend(); plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 17: Test Set Evaluation ──────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        bg = {k: v.to(device) for k, v in batch.items()}
        all_preds.extend(model(bg).cpu().numpy().tolist())
        all_targets.extend(bg['target'].squeeze(1).cpu().numpy().tolist())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

mse  = float(np.mean((all_preds - all_targets) ** 2))
rmse = math.sqrt(mse)
mae  = float(np.mean(np.abs(all_preds - all_targets)))
r2   = float(1 - np.sum((all_preds - all_targets)**2) /
             np.sum((all_targets - all_targets.mean())**2))

print('-' * 44)
print(f'  Test MSE  : {mse:10.2f} h2')
print(f'  Test RMSE : {rmse:10.2f} h')
print(f'  Test MAE  : {mae:10.2f} h')
print(f'  Test R2   : {r2:10.4f}')
print('-' * 44)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_targets, all_preds, alpha=0.3, s=8, label='Patients')
lim = [min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())]
ax.plot(lim, lim, 'r--', lw=1, label='Perfect prediction')
ax.set_xlabel('Actual time to discharge (hours)')
ax.set_ylabel('Predicted time to discharge (hours)')
ax.set_title(f'Test: Predicted vs Actual  (RMSE={rmse:.1f} h, R2={r2:.3f})')
ax.legend(); plt.tight_layout()
plt.savefig('test_scatter.png', dpi=150)
plt.show()